# Phase 6 — Inference & Model Testing

**Course**: CS5143 Natural Language Processing — Spring 2026, FAST-NUCES  
**Student**: Muhammad Azhar (24K-7606)

This notebook loads the trained XLM-R model and runs Named Entity Recognition on
new input sentences. No labelled data needed — just raw text.

Entities recognised:
- **DIS** — Disease (e.g. *diabetes*, *hypertension*)
- **SYM** — Symptom (e.g. *chest pain*, *fever*)
- **PRO** — Procedure (e.g. *MRI*, *coronary artery bypass grafting*)

**Prerequisites**: Complete Phase 4 — `outputs/checkpoints/best_model/` must exist.

## 1. Setup

In [ ]:
import sys
import os
import torch
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from transformers import AutoModelForTokenClassification
from utils import tokenizer, predict, predict_and_print
from dataset import id2label, LABEL_LIST

BEST_MODEL_PATH = REPO_ROOT / 'outputs' / 'checkpoints' / 'best_model'

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Trained model not found at {BEST_MODEL_PATH}.\n'
        'Run Phase 4 training first.'
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'Labels  : {LABEL_LIST}')
print(f'Model   : {BEST_MODEL_PATH}')

## 2. Load the Trained Model

We load the best checkpoint from Phase 4 and switch to `eval()` mode so dropout
layers are disabled and predictions are deterministic.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(str(BEST_MODEL_PATH))
model.to(device).eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded  : {BEST_MODEL_PATH.name}')
print(f'Parameters    : {total_params / 1e6:.1f}M')
print(f'Labels (head) : {model.config.id2label}')

## 3. Single Sentence Test

`predict_and_print()` prints a token-level table with colour-coded labels and
a summary of extracted entity spans below the table.

In [ ]:
# English — try any sentence you like
sentence = "The patient presented with severe chest pain and dyspnea and was diagnosed with heart failure."
predict_and_print(sentence, model, device)

In [ ]:
# Spanish — cross-lingual transfer (no Spanish fine-tuning needed separately)
sentence_es = "El paciente presentó fiebre alta , tos seca y fue diagnosticado con neumonía bacteriana."
predict_and_print(sentence_es, model, device)

## 4. English Sample Sentences — Batch Test

Ten clinically realistic sentences covering all three entity types and a range of
complexity: single-word entities, multi-word spans, and sentences with multiple
entity types. Ideal for checking that all three labels (DIS / SYM / PRO) fire correctly.

In [ ]:
en_samples = [
    # --- Disease examples ---
    "She was diagnosed with type 2 diabetes mellitus and hypertension.",
    "The biopsy confirmed colorectal cancer stage III.",
    "MRI revealed a lesion consistent with multiple sclerosis.",

    # --- Symptom examples ---
    "The child had high fever , persistent cough , and neck stiffness.",
    "He reported sharp epigastric pain , nausea , and vomiting for two days.",

    # --- Procedure examples ---
    "The patient underwent coronary artery bypass grafting after acute myocardial infarction.",
    "Blood cultures were drawn and intravenous antibiotics were administered.",

    # --- Mixed entity types in one sentence ---
    "Echocardiogram revealed mitral valve regurgitation and the patient complained of shortness of breath.",
    "He has chronic obstructive pulmonary disease and requires bronchodilator therapy twice daily.",
    "After appendectomy the patient developed wound infection and sepsis.",
]

for sent in en_samples:
    predict_and_print(sent, model, device)
    print()

## 5. Spanish Sample Sentences — Cross-Lingual Test

XLM-R was trained on 100 languages via a shared SentencePiece vocabulary, so the
model can predict NER labels on Spanish text without any language switching.

In [ ]:
es_samples = [
    "El paciente presentó fiebre alta y tos seca durante tres días.",
    "Se le diagnosticó diabetes mellitus tipo 2 e hipertensión arterial.",
    "La paciente fue sometida a una apendicectomía de urgencia por peritonitis.",
    "Presenta cefalea intensa , náuseas y rigidez de nuca sugestiva de meningitis.",
    "Se inició tratamiento con antibióticos por neumonía bacteriana y el paciente mejoró.",
    "La resonancia magnética mostró una lesión hiperintensa compatible con esclerosis múltiple.",
    "El niño tenía erupción cutánea e inflamación articular compatible con artritis juvenil.",
]

for sent in es_samples:
    predict_and_print(sent, model, device)
    print()

## 6. Interactive Test — Enter Your Own Sentence

Type any clinical sentence (English or Spanish) in the cell below and run it.

In [ ]:
# ── Edit this sentence and re-run the cell ──────────────────────────────────
my_sentence = "Patient has chronic kidney disease and requires dialysis three times a week."
# ────────────────────────────────────────────────────────────────────────────

predict_and_print(my_sentence, model, device)

## 7. Structured Output as a DataFrame

`predict()` returns a list of `{"word", "label"}` dicts. We convert it to a
pandas DataFrame and apply colour highlighting per entity type.

In [ ]:
HIGHLIGHT = {
    "DIS": "background-color: #ffcccc",  # red
    "SYM": "background-color: #fff3cc",  # yellow
    "PRO": "background-color: #ccf5ff",  # cyan
    "O":   "",
}

def ner_dataframe(sentence):
    results = predict(sentence, model, device)
    df = pd.DataFrame(results)
    df["entity_type"] = df["label"].apply(lambda x: x.split("-")[-1] if x != "O" else "O")
    df["bio_prefix"]  = df["label"].apply(lambda x: x.split("-")[0] if "-" in x else "")

    def row_color(row):
        style = HIGHLIGHT.get(row["entity_type"], "")
        return [style] * len(row)

    return df.style.apply(row_color, axis=1)

test_sentence = "The biopsy confirmed colorectal cancer and chemotherapy was initiated."
print("Sentence:", test_sentence)
ner_dataframe(test_sentence)

## 8. Batch Inference with Span-Level Summary Table

Extract all entity spans from a list of sentences and collect them in a single
DataFrame — useful for downstream analysis or reporting.

In [ ]:
def extract_spans(sentence):
    """Return a list of (sentence, span_text, entity_type) tuples."""
    results = predict(sentence, model, device)
    spans = []
    cur_words, cur_type = [], None
    for r in results:
        if r["label"].startswith("B-"):
            if cur_words:
                spans.append((sentence, " ".join(cur_words), cur_type))
            cur_words = [r["word"]]
            cur_type  = r["label"][2:]
        elif r["label"].startswith("I-") and cur_words:
            cur_words.append(r["word"])
        else:
            if cur_words:
                spans.append((sentence, " ".join(cur_words), cur_type))
            cur_words, cur_type = [], None
    if cur_words:
        spans.append((sentence, " ".join(cur_words), cur_type))
    return spans


test_sentences = [
    "She was diagnosed with type 2 diabetes and hypertension after presenting with blurred vision.",
    "The patient underwent MRI and was found to have a herniated disc causing back pain.",
    "He had fever and chills and was started on broad-spectrum antibiotics for sepsis.",
    "Laparoscopic cholecystectomy was performed for acute cholecystitis.",
    "CT scan revealed pulmonary embolism and anticoagulation therapy was initiated.",
]

all_spans = []
for s in test_sentences:
    all_spans.extend(extract_spans(s))

span_df = pd.DataFrame(all_spans, columns=["Sentence", "Entity Span", "Type"])
span_df["Sentence"] = span_df["Sentence"].str[:55] + "..."

print(f"Total spans extracted: {len(span_df)}")
span_df

## 9. Entity Type Distribution Across Test Sentences

Bar chart showing how many DIS / SYM / PRO entities the model extracted from
the batch above — a quick sanity check on model behaviour.

In [ ]:
counts = span_df["Type"].value_counts().reindex(["DIS", "SYM", "PRO"], fill_value=0)
colors = ["#e05c5c", "#e0b84e", "#4ec0e0"]

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(counts.index, counts.values, color=colors, width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(val), ha='center', va='bottom', fontsize=11)

ax.set_title("Predicted Entity Distribution (inference batch)")
ax.set_ylabel("Count")
ax.set_ylim(0, counts.max() + 2)
ax.spines[["top", "right"]].set_visible(False)

legend = [
    mpatches.Patch(color="#e05c5c", label="DIS — Disease"),
    mpatches.Patch(color="#e0b84e", label="SYM — Symptom"),
    mpatches.Patch(color="#4ec0e0", label="PRO — Procedure"),
]
ax.legend(handles=legend, loc="upper right", fontsize=9)
plt.tight_layout()

out_path = REPO_ROOT / 'outputs' / 'results' / 'inference_entity_distribution.png'
plt.savefig(str(out_path), dpi=150, bbox_inches='tight')
print(f"Saved: {out_path}")
plt.show()

## 10. Edge Cases — What the Model May Struggle With

These sentences test known hard cases: rare abbreviations, negated entities,
multi-word procedures, and code-switched text.

In [ ]:
edge_cases = {
    "Abbreviations":
        "Pt c/o SOB and CP , ruled out MI , started on heparin drip.",

    "Negated entity (should NOT be flagged ideally)":
        "The patient denied any chest pain or shortness of breath.",

    "Long multi-word procedure":
        "She underwent percutaneous transluminal coronary angioplasty with stenting.",

    "Multiple entities back-to-back":
        "Fever headache vomiting rash — suspected dengue fever with thrombocytopenia.",

    "Rare entity (OOV stress test)":
        "Diagnosed with Takayasu arteritis and started on immunosuppressive therapy.",

    "Spanish-English code-switch":
        "El patient fue diagnosed con hypertension y diabetes.",
}

for case_name, sent in edge_cases.items():
    print(f"\n>>> {case_name}")
    predict_and_print(sent, model, device)

## Summary

**What this notebook demonstrates**:
- Loading a trained XLM-R checkpoint for token-classification inference
- Running NER on arbitrary English and Spanish clinical text
- Two output modes: colour-coded terminal table (`predict_and_print`) and structured DataFrame (`predict`)
- Batch span extraction with entity distribution chart
- Edge-case analysis: abbreviations, negation, rare entities, code-switching

**Key takeaway**: XLM-R generalises across languages from a single fine-tuned checkpoint — no separate English/Spanish models are needed.

In [ ]:
print("All done — inference notebook ran to completion successfully.")